In [ ]:
import pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'README.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'Data'
OUT  = ROOT / 'Code' / 'output'


# Shock de depósitos a nivel banco

Exposición E_b, Tabla I estilo Gilje-Loutskina-Strahan (2016), grilla visual de las medidas de depósitos alrededor de las reformas y regresiones simples. Insumos: `{DATA}/processed/*.dta`. Salidas: `{OUT}/figuras/sd_*.pdf|png` y `{OUT}/tablas/sd_*.tex`.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
dat = str(DATA / "processed")
fig_dir = str(OUT / "figuras"); tab_dir = str(OUT / "tablas")
os.makedirs(fig_dir, exist_ok=True); os.makedirs(tab_dir, exist_ok=True)
bal = pd.read_stata(os.path.join(dat, "balances_bancos.dta"))
pdp = pd.read_stata(os.path.join(dat, "prestamos_depositos_prov.dta"))
inc = pd.read_stata(os.path.join(dat, "inclusion_prov.dta"))
QR, PULL = pd.Timestamp("2021-11-01"), pd.Timestamp("2022-09-01")
bal = bal[bal.es_banco == 1].copy()
print(bal.shape, pdp.shape, inc.shape)

## 1. Exposición del banco

E_b = Σ_p w_bp × A_p. Pesos: share de cada provincia en los depósitos del banco, promedio de 2021m3/m6/m9 (último pre completo). A_p: personas con cuenta de pago sobre personas con alguna cuenta, 2021q3. Versión alternativa con pesos por sucursales plenas.

In [ ]:
a = inc[inc.ym == "2021-09-01"].copy()
a["adop"] = 100 * a.n_pers_min1_cta_pago / a.n_pers_min1_cta
a["suc_10k"] = 1e4 * a.n_sucursales / a.n_pers_min1_cta
adop = a[["provincia", "adop", "suc_10k"]].set_index("provincia")

pre = pdp[pdp.ym.isin(pd.to_datetime(["2021-03-01", "2021-06-01", "2021-09-01"]))]
w_dep = pre.groupby(["ent", "provincia"], observed=True).depositos_prov.mean().reset_index()
w_dep = w_dep[w_dep.depositos_prov > 0]
w_dep["w"] = w_dep.depositos_prov / w_dep.groupby("ent").depositos_prov.transform("sum")
w_suc = pre.groupby(["ent", "provincia"], observed=True).suc_plenas.mean().reset_index()
w_suc = w_suc[w_suc.suc_plenas > 0]
w_suc["w"] = w_suc.suc_plenas / w_suc.groupby("ent").suc_plenas.transform("sum")

def expo(w):
    m = w.merge(adop.adop, left_on="provincia", right_index=True)
    out = m.assign(wA=m.w * m.adop).groupby("ent").agg(E=("wA", "sum"), n_prov=("w", "size"))
    caba = m[m.provincia == "CABA"].set_index("ent").w.rename("w_caba")
    return out.join(caba).fillna({"w_caba": 0})
E = expo(w_dep).rename(columns={"E": "E_dep"}).join(expo(w_suc).E.rename("E_suc"))
names = bal.sort_values("ym").groupby("ent").nombre.last()
E = E.join(names).dropna(subset=["E_dep"])
E = E[E.index.isin(bal.ent.unique())]
E["tercil"] = pd.qcut(E.E_dep, 3, labels=[1, 2, 3]).astype(int)
print("bancos con E_b:", len(E), "| corr(E_dep, E_suc) =", round(E.E_dep.corr(E.E_suc), 3))
E.sort_values("E_dep", ascending=False)[["nombre", "E_dep", "E_suc", "n_prov", "w_caba", "tercil"]].round(3).to_csv(os.path.join(tab_dir, "sd_exposicion_ranking.csv"))
E.sort_values("E_dep", ascending=False)[["nombre", "E_dep", "E_suc", "n_prov", "w_caba"]].round(2).head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.hist(E.E_dep, bins=18, color="navy", alpha=0.7, label="pesos por depositos")
ax.hist(E.E_suc, bins=18, color="maroon", alpha=0.5, label="pesos por sucursales")
ax.set_xlabel("E_b: adopcion provincial ponderada, %", fontsize=9); ax.set_ylabel("Bancos", fontsize=9)
ax.legend(fontsize=8, frameon=False); ax.tick_params(labelsize=8); fig.tight_layout()
fig.savefig(os.path.join(fig_dir, "sd_exposicion_hist.png"), dpi=150); plt.close(fig)

## 2. Tabla I estilo Gilje

Panel A: provincias (24). Panel B: exposición del banco. Panel C: características del banco, promedio del pre (2021m1-2021m10), tercil alto vs bajo de E_b. Panel D: banco × provincia × trimestre. Formato: media y desvío por grupo.

In [ ]:
def fmt(x, d=2):
    return "" if pd.isna(x) else f"{x:,.{d}f}"
def panel_rows(df, vars_labels, by):
    rows = []
    for v, lab in vars_labels:
        g = df.groupby(by)[v]
        m, s = g.mean(), g.std()
        rows.append((lab, fmt(m.get(3)), fmt(s.get(3)), fmt(m.get(1)), fmt(s.get(1)), fmt(m.get(3) - m.get(1))))
    return rows

# Panel A: provinces
pa = adop.copy()
dep_prov = pre.groupby("provincia", observed=True).depositos_prov_r.sum() / 3
pa["dep_pc"] = dep_prov / a.set_index("provincia").n_pers_min1_cta / 1e3
pa["terc"] = pd.qcut(pa.adop, 3, labels=[1, 2, 3]).astype(int)
A = panel_rows(pa, [("adop", "Adopcion de billeteras (\\%)"), ("suc_10k", "Sucursales por 10 mil personas con cuenta"), ("dep_pc", "Depositos reales por persona con cuenta (miles)")], "terc")

# Panel B: bank exposure
B = panel_rows(E, [("E_dep", "E_b, pesos por depositos"), ("E_suc", "E_b, pesos por sucursales"), ("n_prov", "Provincias con presencia"), ("w_caba", "Share de depositos en CABA")], "tercil")

# Panel C: bank characteristics, pre window
b = bal.merge(E[["E_dep", "tercil"]], left_on="ent", right_index=True)
b = b.sort_values(["ent", "ym"])
b["dlog_dep"] = b.groupby("ent").dep_spnf_r.transform(lambda s: np.log(s).diff(12))
b["log_activo"] = np.log(b.activo_r)
b["dep_activo"] = b.depositos / b.activo
b["costo_dep"] = 12 * b.int_dep_ars_flow / b.dep_spnf
b["tasa_ca"] = 12 * b.int_cajaahorro_flow / b.dep_cajaahorro.replace(0, np.nan)
b["liq_activo"] = (b.efectivo + b.titulos) / b.activo
b["pn_activo"] = b.pn / b.activo
b["prest_activo"] = b.prest_spnf / b.activo
b["roa"] = b.resultados / b.activo
for v in ["dep_cajaahorro", "dep_vista_min", "dep_ctacte", "dep_pf_total", "dep_ctaesp_cint", "dep_spnf_usd"]:
    b["sh_" + v] = b[v] / (b.dep_spnf + b.dep_spnf_usd)
pre_b = b[(b.ym >= "2021-01-01") & (b.ym <= "2021-10-01")].groupby("ent").mean(numeric_only=True)
pre_b["tercil"] = E.tercil
C = panel_rows(pre_b, [
    ("dlog_dep", "Crecimiento real de depositos SPNF, 12 meses"),
    ("log_activo", "Log activo real"), ("dep_activo", "Depositos / activo"),
    ("costo_dep", "Costo de los depositos en pesos (intereses / depositos, anualizado)"),
    ("tasa_ca", "Tasa implicita caja de ahorros (anualizada)"),
    ("liq_activo", "Liquidez (efectivo + titulos) / activo"), ("pn_activo", "Patrimonio / activo"),
    ("prest_activo", "Prestamos SPNF / activo"), ("roa", "Resultados / activo"),
    ("sh_dep_cajaahorro", "Share caja de ahorros"), ("sh_dep_vista_min", "Share vista minorista"),
    ("sh_dep_ctacte", "Share cuenta corriente sin interes"), ("sh_dep_pf_total", "Share plazo fijo"),
    ("sh_dep_ctaesp_cint", "Share cta cte especial PJ con interes"), ("sh_dep_spnf_usd", "Share depositos en dolares")], "tercil")

# Panel D: bank x province x quarter
q = pdp[pdp.ym.dt.month.isin([3, 6, 9, 12]) & (pdp.depositos_prov > 0)].copy()
q = q.merge(pa[["terc"]], left_on="provincia", right_index=True)
q = q.sort_values(["ent", "provincia", "ym"])
q["g_dep"] = q.groupby(["ent", "provincia"], observed=True).depositos_prov_r.transform(lambda s: np.log(s).diff(4))
q["g_pre"] = q.groupby(["ent", "provincia"], observed=True).prestamos_prov_r.transform(lambda s: np.log(s.replace(0, np.nan)).diff(4))
qpre = q[(q.ym >= "2019-01-01") & (q.ym <= "2021-09-01")]
D = panel_rows(qpre, [("g_dep", "Crecimiento real anual de depositos banco-provincia"), ("g_pre", "Crecimiento real anual de prestamos banco-provincia"), ("suc_plenas", "Sucursales plenas del banco en la provincia")], "terc")
nD = f"N celdas banco-provincia-trimestre: {len(qpre):,}; bancos: {qpre.ent.nunique()}; provincias: 24"

def tex_table(path, panels, title):
    L = ["\\begin{table}[htbp]\\centering", f"\\caption{{{title}}}", "\\small", "\\begin{tabular}{lrrrrr}", "\\toprule",
         " & \\multicolumn{2}{c}{Exposicion alta} & \\multicolumn{2}{c}{Exposicion baja} & \\\\",
         "\\cmidrule(lr){2-3}\\cmidrule(lr){4-5}", " & Media & Desvio & Media & Desvio & Diferencia \\\\", "\\midrule"]
    for name, rows, note in panels:
        L.append(f"\\multicolumn{{6}}{{l}}{{\\textit{{{name}}}}} \\\\")
        for r in rows: L.append(" & ".join(r) + " \\\\")
        if note: L.append(f"\\multicolumn{{6}}{{l}}{{\\footnotesize {note}}} \\\\")
        L.append("\\addlinespace")
    L += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
    open(path, "w").write("\n".join(L))
tex_table(os.path.join(tab_dir, "sd_tabla1.tex"), [
    ("Panel A. Provincias, 2021q3 (terciles de adopcion)", A, "24 provincias; exposicion alta = tercil 3 de adopcion, baja = tercil 1"),
    ("Panel B. Exposicion del banco (terciles de E_b)", B, f"{len(E)} bancos"),
    ("Panel C. Caracteristicas del banco, promedio 2021m1-2021m10", C, ""),
    ("Panel D. Banco x provincia x trimestre, 2019q1-2021q3 (terciles de adopcion de la provincia)", D, nD)],
    "Estadisticas descriptivas")
pd.DataFrame(C, columns=["", "media alta", "sd alta", "media baja", "sd baja", "dif"])

## 3. La grilla: medidas chicas de depósitos alrededor de cada reforma

Cada medida real, índice 100 en el mes previo a la reforma, suma por tercil de exposición (alto vs bajo), ventana ±12 meses. Última fila: precio, tasa implícita de caja de ahorros.

In [ ]:
measures = [("dep_cajaahorro_r", "Caja de ahorros"), ("dep_vista_min_r", "Vista minorista"), ("dep_ctacte_r", "Cta cte sin interes"), ("dep_pf_total_r", "Plazo fijo total"), ("dep_ctaesp_cint_r", "Cta cte esp PJ con interes"), ("dep_spnf_r", "Total SPNF pesos")]
def grid(b, shocks, fname):
    fig, axes = plt.subplots(len(measures) + 1, len(shocks), figsize=(5.2 * len(shocks), 2.3 * (len(measures) + 1)), sharex="col")
    for j, (sh, lab) in enumerate(shocks):
        base_m = sh - pd.DateOffset(months=1)
        win = b[(b.ym >= sh - pd.DateOffset(months=12)) & (b.ym <= sh + pd.DateOffset(months=12))]
        for i, (v, vlab) in enumerate(measures):
            ax = axes[i, j]
            g = win.groupby(["tercil", "ym"], observed=True)[v].sum().reset_index()
            for t, col in [(1, "navy"), (3, "maroon")]:
                d = g[g.tercil == t].set_index("ym")[v]
                ax.plot(d.index, 100 * d / d.loc[base_m], color=col, lw=1.6, label=f"Tercil {t}")
            ax.axvline(sh, color="black", lw=1); ax.set_ylabel(vlab, fontsize=8); ax.tick_params(labelsize=7)
            if i == 0: ax.set_title(lab, fontsize=10)
            if i == 0 and j == 0: ax.legend(fontsize=7, frameon=False)
        ax = axes[len(measures), j]
        g = win.groupby(["tercil", "ym"], observed=True)[["int_cajaahorro_flow", "dep_cajaahorro"]].sum().reset_index()
        g["tasa"] = 100 * 12 * g.int_cajaahorro_flow / g.dep_cajaahorro
        for t, col in [(1, "navy"), (3, "maroon")]:
            d = g[g.tercil == t].set_index("ym").tasa
            ax.plot(d.index, d, color=col, lw=1.6)
        ax.axvline(sh, color="black", lw=1); ax.set_ylabel("Tasa implicita CA, % anual", fontsize=8); ax.tick_params(labelsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(fig_dir, fname), dpi=150); plt.close(fig)
grid(b, [(QR, "QR interoperable, 2021m11"), (PULL, "Pull, 2022m9")], "sd_grilla_medidas.png")

## 4. Regresiones simples

Δlog y_bt (12 meses, real) sobre E_b × Post, para cada medida y cada shock. Tres columnas por medida: (1) FE banco + mes, sin controles; (2) + controles rezagados básicos (log activo, depósitos/activo, liquidez/activo, PN/activo); (3) FE banco + tipo de banco × mes. E_b estandarizada. Errores estándar convencionales por ahora.

In [ ]:
import statsmodels.formula.api as smf
b["E_std"] = (b.E_dep - b.E_dep.mean()) / b.E_dep.std()
b["tipo"] = np.select([b.banco_publico == 1, b.banco_capextranjero == 1], ["publico", "extranjero"], "privado_nac")
for v in ["dep_cajaahorro_r", "dep_vista_min_r", "dep_ctacte_r", "dep_pf_total_r", "dep_ctaesp_cint_r", "dep_spnf_r"]:
    b["dl_" + v] = b.groupby("ent")[v].transform(lambda s: np.log(s.replace(0, np.nan)).diff(12))
for c in ["log_activo", "dep_activo", "liq_activo", "pn_activo"]:
    b[c + "_l12"] = b.groupby("ent")[c].shift(12)
b["ym_f"] = b.ym.dt.strftime("%Y%m"); b["ent_f"] = b.ent.astype(int).astype(str)
b["tipo_ym"] = b.tipo + "_" + b.ym_f

def run(shock, lab, pre_m=12, post_m=12):
    win = b[(b.ym >= shock - pd.DateOffset(months=pre_m)) & (b.ym <= shock + pd.DateOffset(months=post_m))].copy()
    win["post"] = (win.ym >= shock).astype(int)
    win["ExP"] = win.E_std * win.post
    rows = []
    for v, vlab in measures:
        y = "dl_" + v
        specs = {"(1) FE banco+mes": f"{y} ~ ExP + C(ent_f) + C(ym_f)",
                 "(2) + controles": f"{y} ~ ExP + log_activo_l12 + dep_activo_l12 + liq_activo_l12 + pn_activo_l12 + C(ent_f) + C(ym_f)",
                 "(3) FE banco + tipo x mes": f"{y} ~ ExP + C(ent_f) + C(tipo_ym)"}
        for sname, f in specs.items():
            d = win.dropna(subset=[y, "ExP"])
            r = smf.ols(f, data=d).fit()
            rows.append({"shock": lab, "medida": vlab, "spec": sname, "beta": r.params["ExP"], "se": r.bse["ExP"], "t": r.tvalues["ExP"], "N": int(r.nobs)})
    return pd.DataFrame(rows)
res = pd.concat([run(QR, "QR 2021m11"), run(PULL, "Pull 2022m9")])
res.round(4).to_csv(os.path.join(tab_dir, "sd_reg_simple.csv"), index=False)
res.pivot_table(index=["shock", "medida"], columns="spec", values="beta").round(3)

In [ ]:
# event study, monthly coefficients, spec (1), two measures
def event(shock, v, pre_m=12, post_m=12):
    win = b[(b.ym >= shock - pd.DateOffset(months=pre_m)) & (b.ym <= shock + pd.DateOffset(months=post_m))].copy()
    win["k"] = ((win.ym.dt.year - shock.year) * 12 + win.ym.dt.month - shock.month).astype(int)
    y = "dl_" + v
    d = win.dropna(subset=[y])
    ks = sorted(d.k.unique()); base = -1
    terms = " + ".join([f"I((k == {k}) * E_std)" for k in ks if k != base])
    r = smf.ols(f"{y} ~ {terms} + C(ent_f) + C(ym_f)", data=d).fit()
    out = pd.DataFrame({"k": [k for k in ks if k != base], "b": [r.params[f"I((k == {k}) * E_std)"] for k in ks if k != base], "se": [r.bse[f"I((k == {k}) * E_std)"] for k in ks if k != base]})
    return pd.concat([out, pd.DataFrame({"k": [base], "b": [0.0], "se": [0.0]})]).sort_values("k")
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
for j, (sh, lab) in enumerate([(QR, "QR 2021m11"), (PULL, "Pull 2022m9")]):
    for i, (v, vlab) in enumerate([("dep_cajaahorro_r", "Caja de ahorros"), ("dep_vista_min_r", "Vista minorista")]):
        e = event(sh, v); ax = axes[i, j]
        ax.errorbar(e.k, e.b, yerr=1.96 * e.se, fmt="o", color="navy", ms=3, capsize=2, lw=1)
        ax.axhline(0, color="black", lw=0.8); ax.axvline(-0.5, color="black", lw=0.8, ls="--")
        ax.set_title(f"{vlab}, {lab}", fontsize=10); ax.tick_params(labelsize=8)
        if j == 0: ax.set_ylabel("Coef. E_b (std) x mes", fontsize=9)
        if i == 1: ax.set_xlabel("Meses desde la reforma", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "sd_event_study.png"), dpi=150); plt.close(fig)